# Deep Learning Foundations, ANNs, and Polynomial Regression

## Exercise 1: Deep Learning vs. Traditional Machine Learning

### Comparative Table

In [ ]:
import pandas as pd

comparison = pd.DataFrame({
    "Aspect": [
        "Feature Engineering",
        "Data Processing",
        "Scalability",
        "Pattern Discovery",
        "Computational Requirements"
    ],
    "Traditional Machine Learning": [
        "Manual; requires domain expertise to extract relevant features",
        "Works well with structured, tabular data",
        "Performance plateaus with very large datasets",
        "Learns patterns from hand-crafted features",
        "Low to moderate; runs on standard CPUs"
    ],
    "Deep Learning": [
        "Automatic; learns features directly from raw data",
        "Excels with unstructured data (images, text, audio)",
        "Performance keeps improving with more data",
        "Learns hierarchical representations automatically",
        "High; typically requires GPUs/TPUs for training"
    ]
})

comparison

### Real-World Problem Examples

**Traditional ML is better suited**: Predicting customer churn from structured tabular data (account balance, tenure, number of products). The features are already well-defined and limited in number, so a model like Logistic Regression or Random Forest can achieve strong performance quickly, with full interpretability of which factors drive churn.

**Deep Learning is the superior choice**: Image classification for medical diagnosis (e.g., detecting tumors in MRI scans). The raw pixel data has no obvious hand-craftable features, and the spatial patterns relevant to diagnosis are too complex to engineer manually. A Convolutional Neural Network can automatically learn the relevant visual features directly from the images.

### Why Deep Learning Excels at Unstructured Data

Deep learning models, particularly neural networks with multiple layers, can automatically learn hierarchical feature representations directly from raw unstructured data such as images, text, or audio. Unlike traditional machine learning, which depends on humans manually defining relevant features, deep learning layers progressively extract increasingly abstract patterns, starting from simple edges or characters and building up to complex shapes or semantic meaning. This is especially valuable for unstructured data because the relevant features are often too numerous, too subtle, or too interdependent for a human expert to define explicitly. As a result, deep learning models tend to outperform traditional ML on tasks like image recognition, natural language understanding, and speech processing, provided sufficient data and computational resources are available.

## Exercise 2: Artificial Neural Networks (ANNs)

### ANN Diagram: 3 Input Neurons, 4 Hidden Neurons, 2 Output Neurons

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

fig, ax = plt.subplots(figsize=(10, 7))
ax.set_xlim(0, 10)
ax.set_ylim(0, 8)
ax.axis("off")

layer_x = {"input": 1.5, "hidden": 5, "output": 8.5}

input_y = [6, 4, 2]
hidden_y = [6.5, 5, 3.5, 2]
output_y = [5, 3]

# Draw connections (weights) input -> hidden
for iy in input_y:
    for hy in hidden_y:
        ax.plot([layer_x["input"], layer_x["hidden"]], [iy, hy],
                color="#bbbbbb", lw=0.8, zorder=1)

# Draw connections hidden -> output
for hy in hidden_y:
    for oy in output_y:
        ax.plot([layer_x["hidden"], layer_x["output"]], [hy, oy],
                color="#bbbbbb", lw=0.8, zorder=1)

# Draw neurons
for iy in input_y:
    ax.add_patch(plt.Circle((layer_x["input"], iy), 0.35,
                             color="#5B9BD5", ec="black", zorder=2))
for hy in hidden_y:
    ax.add_patch(plt.Circle((layer_x["hidden"], hy), 0.35,
                             color="#70AD47", ec="black", zorder=2))
for oy in output_y:
    ax.add_patch(plt.Circle((layer_x["output"], oy), 0.35,
                             color="#ED7D31", ec="black", zorder=2))

# Layer labels
ax.text(layer_x["input"], 7.2, "Input Layer\n(3 neurons)",
        ha="center", fontsize=11, fontweight="bold")
ax.text(layer_x["hidden"], 7.5, "Hidden Layer\n(4 neurons)",
        ha="center", fontsize=11, fontweight="bold")
ax.text(layer_x["output"], 6, "Output Layer\n(2 neurons)",
        ha="center", fontsize=11, fontweight="bold")

# Component annotations
ax.annotate("Weight (connection)", xy=(3.2, 5.2), xytext=(2.5, 0.8),
            fontsize=9, color="#444444",
            arrowprops=dict(arrowstyle="->", color="#444444", lw=1))

ax.text(layer_x["hidden"], 0.8,
        "Each hidden/output neuron applies:\noutput = activation(sum(weights × inputs) + bias)",
        ha="center", fontsize=9, style="italic")

plt.title("Simple Artificial Neural Network (3-4-2 Architecture)", fontsize=13)
plt.tight_layout()
plt.savefig("ann_diagram.png", dpi=150, bbox_inches="tight")
plt.show()

### How Information Flows Through the Network

Information enters through the input layer, where each of the 3 neurons holds one feature value from the data. Each connection between neurons carries a weight that scales the signal, and each neuron in the hidden and output layers adds a bias term before applying an activation function (such as ReLU or sigmoid), which introduces non-linearity. The weighted, biased sum from all 3 input neurons feeds into each of the 4 hidden neurons, and this process repeats from the hidden layer to the 2 output neurons. This sequence of weighted sums and activations is called forward propagation, and it transforms the raw input into the network's final prediction.

## Exercise 3: Creating the Dataset and Visualizing the Data

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error

np.random.seed(0)
x = np.arange(-1, 1, 0.1)
y = -x**2 + np.random.normal(0, 0.05, len(x))

print(f"Number of points: {len(x)}")
print(f"x values: {x}")
print(f"y values: {y}")

In [ ]:
plt.figure(figsize=(7, 5))
plt.scatter(x, y, color="steelblue", s=50, edgecolor="black", zorder=3)
plt.plot(x, -x**2, color="gray", linestyle="--", label="True function y = -x²")
plt.title("Noisy Dataset: y = -x² + noise")
plt.xlabel("x")
plt.ylabel("y")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Split: first 12 points = training set, last 8 points = test set
x_train, y_train = x[:12], y[:12]
x_test, y_test = x[12:], y[12:]

print(f"Training set size: {len(x_train)}")
print(f"Test set size:     {len(x_test)}")

plt.figure(figsize=(7, 5))
plt.scatter(x_train, y_train, color="steelblue", s=50,
            edgecolor="black", label="Training set", zorder=3)
plt.scatter(x_test, y_test, color="tomato", s=50,
            edgecolor="black", label="Test set", zorder=3)
plt.title("Train/Test Split")
plt.xlabel("x")
plt.ylabel("y")
plt.legend()
plt.tight_layout()
plt.show()

## Exercise 4: Fitting Polynomial Models of Different Degrees

In [ ]:
def polynomial_fit(degree, x_train, y_train):
    """
    Fits a polynomial of the given degree to the training data.

    Parameters:
        degree  : int, degree of the polynomial
        x_train : array of training x values
        y_train : array of training y values

    Returns:
        numpy.poly1d object representing the fitted polynomial
    """
    coefficients = np.polyfit(x_train, y_train, degree)
    return np.poly1d(coefficients)


def plot_polyfit(degree, x_train, y_train, x_test, y_test):
    """
    Plots the training set, test set, and the fitted polynomial curve
    for a given polynomial degree.
    """
    poly = polynomial_fit(degree, x_train, y_train)

    x_curve = np.linspace(-1, 1, 200)
    y_curve = poly(x_curve)

    plt.figure(figsize=(7, 5))
    plt.scatter(x_train, y_train, color="steelblue", s=50,
                edgecolor="black", label="Training set", zorder=3)
    plt.scatter(x_test, y_test, color="tomato", s=50,
                edgecolor="black", label="Test set", zorder=3)
    plt.plot(x_curve, y_curve, color="mediumseagreen", lw=2,
              label=f"Degree {degree} fit")

    train_rmse = np.sqrt(mean_squared_error(y_train, poly(x_train)))
    test_rmse  = np.sqrt(mean_squared_error(y_test, poly(x_test)))

    plt.title(f"Polynomial Degree {degree}\nTrain RMSE: {train_rmse:.4f} | Test RMSE: {test_rmse:.4f}")
    plt.xlabel("x")
    plt.ylabel("y")
    plt.ylim(-1.5, 1.5)
    plt.legend()
    plt.tight_layout()
    plt.show()

    return train_rmse, test_rmse


print("Functions defined: polynomial_fit, plot_polyfit")

In [ ]:
for degree in [1, 7, 11]:
    plot_polyfit(degree, x_train, y_train, x_test, y_test)

**Observations**

- **Degree 1** (linear): The model underfits — a straight line cannot capture the curvature of y = -x², resulting in high error on both training and test sets.
- **Degree 7**: The model fits the training data reasonably well and starts to approximate the true quadratic shape, though it may begin showing slight wiggles between points.
- **Degree 11**: The model overfits dramatically — with only 12 training points, an 11th-degree polynomial can pass through (or very close to) every training point, producing wild oscillations between points. Training error becomes very low, but test error increases sharply, demonstrating classic overfitting.

## Exercise 5: Cross-Validation to Find the Optimal Degree

In [ ]:
results = []

for degree in range(1, 12):
    poly = polynomial_fit(degree, x_train, y_train)

    train_rmse = np.sqrt(mean_squared_error(y_train, poly(x_train)))
    test_rmse  = np.sqrt(mean_squared_error(y_test, poly(x_test)))

    results.append({
        "degree": degree,
        "train_rmse": train_rmse,
        "test_rmse": test_rmse
    })

results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(results_df["degree"], results_df["train_rmse"],
         marker="o", color="steelblue", label="Training RMSE")
plt.plot(results_df["degree"], results_df["test_rmse"],
         marker="s", color="tomato", label="Test RMSE")
plt.yscale("log")
plt.xlabel("Polynomial Degree")
plt.ylabel("RMSE (log scale)")
plt.title("Training vs Test RMSE by Polynomial Degree")
plt.legend()
plt.grid(True, which="both", alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
best_degree = results_df.loc[results_df["test_rmse"].idxmin(), "degree"]
best_test_rmse = results_df["test_rmse"].min()

print(f"Optimal polynomial degree (lowest test RMSE): {best_degree}")
print(f"Test RMSE at optimal degree: {best_test_rmse:.4f}")
print(f"\nTrue underlying model: y = -x^2  (degree 2)")

if best_degree == 2:
    print("\nConfirmed: the cross-validation result matches the true underlying model.")
else:
    print(f"\nNote: cross-validation selected degree {best_degree}, close to the true degree 2,")
    print("which is expected given the small sample size and added noise.")

**Conclusion**

The RMSE curve typically shows training error decreasing monotonically as the polynomial degree increases, since higher-degree polynomials have more flexibility to fit the training points exactly. Test error, however, follows a U-shape: it decreases initially as the model captures the true quadratic relationship, reaches a minimum around degree 2, and then increases sharply for higher degrees as the model overfits the noise in the training data rather than the underlying signal. This experiment illustrates why cross-validation is essential for model selection: relying on training error alone would lead us to choose an overly complex model (degree 11) that performs poorly on unseen data, while test-set performance correctly identifies the simpler model that matches the true data-generating process.